In [1]:
import numpy as np
import torch
import json 

from utils.pos_encoding import get_pos_encoding

In [2]:
json_path = '/mnt/d/ML/Kaggle/StarWars/episode-76554885.json'
with open(json_path, 'r') as f:
    logs = json.load(f)
    print(logs.keys())


dict_keys(['configuration', 'description', 'id', 'info', 'module_version', 'name', 'rewards', 'schema_version', 'specification', 'statuses', 'steps', 'title', 'version'])


## Start

In [3]:
idx = 0
fleets = np.array(logs['steps'][idx][0]['observation']['fleets'])
planets = np.array(logs['steps'][idx][0]['observation']['planets'])
comet_planet_ids = np.array(logs['steps'][idx][0]['observation']['comet_planet_ids'])
angular_velocity = logs['steps'][idx][0]['observation']['angular_velocity']
initial_planets = np.array(logs['steps'][idx][0]['observation']['initial_planets'])

planets.shape, fleets.shape,  comet_planet_ids.shape, angular_velocity, initial_planets.shape

((32, 7), (0,), (0,), 0.032381599507288125, (32, 7))

In [4]:
encoder = Encoder()
state, mask = encoder.encode(planets, fleets, initial_planets, 
                             angular_velocity, comet_planet_ids, time_step=120, apply_padding=True)

state.shape

NameError: name 'Encoder' is not defined

In [11]:
mask

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [13]:

q_network = Q_network()

action = torch.zeros(5, q_network.d_model)  # dummy actions, shape: [num_actions, d_model]
value = q_network(state, action)
value.shape

/home/tan/miniconda3/envs/arc/lib/python3.10/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


torch.Size([1])

In [16]:
planets, fleets = encoder.convert(planets, fleets, initial_planets, angular_velocity, comet_planet_ids)
planets.shape, fleets.shape

((32, 10), (5, 8))

In [5]:
v_network = V_network()
value = v_network(state)
value

tensor([-0.5240], grad_fn=<SqueezeBackward1>)

In [6]:
policy = P_network()
mu, sigma = policy(state)
mu.shape, sigma.shape

(torch.Size([92, 1, 8]), torch.Size([92, 1, 8]))

In [7]:
action_decoder = ActionDecoder(d_action=8)
action = action_decoder(mu, sigma)
action.shape

/mnt/d/ML/Kaggle/StarWars/501-Legion/model/SAC.py:294: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  action = torch.mm(action, action.T)  # shape: [num_planets, num_planets]


RuntimeError: self must be a matrix

In [9]:
#create batched inputs 
# max num of planets is 40, max fleets is 200, max comet_planet_ids is 10, angular_velocity is scalar, initial_planets is 40
idx_range = range(0, 10)

#set to max values for padding

batched_fleets = np.array([np.array(logs['steps'][i][0]['observation']['fleets']) for i in idx_range])

batched_planets = np.array([np.array(logs['steps'][i][0]['observation']['planets']) for i in idx_range])
batched_comet_planet_ids = np.array([np.array(logs['steps'][i][0]['observation']['comet_planet_ids']) for i in idx_range])
batched_angular_velocity = np.array([logs['steps'][i][0]['observation']['angular_velocity'] for i in idx_range])
batched_initial_planets = np.array([np.array(logs['steps'][i][0]['observation']['initial_planets']) for i in idx_range])

#print shapes
print("Batched fleets shape:", batched_fleets.shape)
print("Batched planets shape:", batched_planets.shape)      
print("Batched comet_planet_ids shape:", batched_comet_planet_ids.shape)
print("Batched angular_velocity shape:", batched_angular_velocity.shape)
print("Batched initial_planets shape:", batched_initial_planets.shape)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (10,) + inhomogeneous part.

## -------------------------------------

In [5]:
class Encoder:
    def __init__(self, max_planets=40, max_fleets=100, max_comet_planet_ids=10, max_speed=6.0):  
        self.max_planets = max_planets
        self.max_fleets = max_fleets 
        self.max_comet_planet_ids = max_comet_planet_ids
        self.max_speed = max_speed
        self.sun = np.array([50, 50])
        #limits of the box in which planets can move
        self.box = [(0, 100),  (0, 100)]
        self.max_time_step = 500
    
    def get_speed(self, ships):
        return 1.0 + (self.max_speed - 1.0) * (np.log(ships) / np.log(1000)) ** 1.5
    
    def get_planet_mapping(self, initial_planets):
        sun = self.sun
        oribital_radius = ((initial_planets[:, 2:4] - sun)**2).sum(axis=1)**0.5
        planet_radius = initial_planets[:, 4]
        moving_planets = oribital_radius + planet_radius < 50

        planet_ids = initial_planets[:, 0]
        planets_mapping = {}
        for i, planet_id in enumerate(planet_ids):
            planets_mapping[int(planet_id)] = int(moving_planets[i])

        return planets_mapping
    
    def _pad_array(self, arr, max_size):
        current_size = arr.shape[0]
        feature_dim = arr.shape[1]
        
        padded_arr = np.zeros((max_size, feature_dim))
        mask = np.zeros(max_size)
        
        # Copy real data
        padded_arr[:current_size] = arr
        mask[:current_size] = 1
        
        return padded_arr, mask

    def encode(self, planets, fleets, initial_planets, angular_velocity, comet_planet_ids, time_step, apply_padding=True):
        # planets - [id, owner, x, y, radius, ships, production]
        # out - [owner, radius, ships, production, moving/static, angular_velocity, comet]
        # fleets - [id, owner, x, y, angle, from_planet_id, ships] or empty array
        # out - [owner, angle, from_planet_id, ships, speed] or empty
        
        #assert the shape of planets
        assert planets.shape[-1] == 7, f"Expected planets to have 7 features, but got {planets.shape[-1]}"
        
        # Handle empty fleets array
        if fleets.size == 0:
            fleets = np.empty((0, 7))
        else:
            assert fleets.shape[-1] == 7, f"Expected fleets to have 7 features, but got {fleets.shape[-1]}"

        planets_owner = planets[:, 1].astype(int)
        planet_pos = planets[:, 2:4]
        
        angular_velocity = np.array([angular_velocity for _ in range(planets.shape[0])]).reshape(-1, 1)
        
        # Handle empty comet_planet_ids
        if comet_planet_ids.size == 0:
            comet_planet = np.zeros((planets.shape[0], 1))
        else:
            comet_planet = np.array([1 if planet_id in comet_planet_ids else 0 for planet_id in planets[:, 0]]).reshape(-1, 1)
        
        planets_mapping = self.get_planet_mapping(initial_planets)
        moving_planets = np.array([planets_mapping.get(planet_id, 0) for planet_id in planets[:, 0]]).reshape(-1, 1)
        #replace nan with 0
        moving_planets = np.nan_to_num(moving_planets)

        planets_owner_one_hot = np.zeros((planets_owner.shape[0], 4))
        for i in range(planets_owner.shape[0]):
            if planets_owner[i] != -1:
                planets_owner_one_hot[i] = np.eye(4)[planets_owner[i]]
        
        #adding time step as feature to planets and fleets
        times_step_array = np.array([time_step for _ in range(planets.shape[0])]).reshape(-1, 1)
        is_planet = np.zeros((planets.shape[0], 1))
        planets_encoded = np.hstack((planets_owner_one_hot, planets[:, 4:], moving_planets, angular_velocity, 
                             comet_planet, is_planet, planet_pos,  times_step_array))
        
        # Handle empty fleets
        if fleets.shape[0] == 0:
            # Create empty fleets_encoded with correct feature dimension (14 features)
            fleets_encoded = np.empty((0, 14))
        else:
            #fleets
            fleets_owner = fleets[:, 1].astype(int)
            fleets_angle = fleets[:, 4].reshape(-1, 1)
            # fleets_from_planet_id = fleets[:, 5].reshape(-1, 1)
            fleets_ships = fleets[:, 6].reshape(-1, 1)
            fleets_speed = np.array([self.get_speed(ships) for ships in fleets_ships[:, 0]]).reshape(-1, 1)

            fleets_owner_one_hot = np.zeros((fleets_owner.shape[0], 4))
            for i in range(fleets_owner.shape[0]):
                if fleets_owner[i] != -1:
                    fleets_owner_one_hot[i] = np.eye(4)[fleets_owner[i]]

            dummy = np.zeros((fleets.shape[0], 3))
            is_fleet = np.ones((fleets.shape[0], 1))
            fleet_pos = fleets[:, 2:4]
            times_step_array_fleet = np.array([time_step for _ in range(fleets.shape[0])]).reshape(-1, 1)

            fleets_encoded = np.hstack((fleets_owner_one_hot, fleets_angle, fleets_ships, fleets_speed, dummy,
                                is_fleet,  fleet_pos, times_step_array_fleet))

        # Apply padding if requested
        if apply_padding:
            planets_encoded, planets_mask = self._pad_array(planets_encoded, self.max_planets)
            fleets_encoded, fleets_mask = self._pad_array(fleets_encoded, self.max_fleets)
            
            # Combine masks
            mask = np.concatenate([planets_mask, fleets_mask])
            
            state = np.vstack((planets_encoded, fleets_encoded))
            return state, mask
        else:
            state = np.vstack((planets_encoded, fleets_encoded))
            return state, None
    
    def encode_batch(self, batched_planets, batched_fleets, batched_initial_planets, batched_angular_velocity, 
                     batched_comet_planet_ids, batch_time_steps=None, apply_padding=True):
        """
        Encode a batch of states.
        
        Args:
            batched_planets: list of planet arrays, each [n_planets, 7]
            batched_fleets: list of fleet arrays, each [n_fleets, 7] or empty
            batched_initial_planets: list of initial planet arrays, each [n_initial, 7]
            batched_angular_velocity: array or list of angular velocities, shape [batch_size]
            batched_comet_planet_ids: list of comet planet id arrays or empty
            batch_time_steps: array/list of time steps for each batch item, or single value for all
            apply_padding: whether to apply padding and return masks
            
        Returns:
            batch_states: [batch_size, max_planets + max_fleets, feature_dim]
            batch_masks: [batch_size, max_planets + max_fleets] if apply_padding=True, else None
        """
        batch_size = len(batched_planets)
        
        # Handle time steps
        if batch_time_steps is None:
            batch_time_steps = [100] * batch_size
        elif isinstance(batch_time_steps, (int, float)):
            batch_time_steps = [batch_time_steps] * batch_size
        
        batch_states = []
        batch_masks = []
        
        for i in range(batch_size):
            state, mask = self.encode(
                batched_planets[i],
                batched_fleets[i],
                batched_initial_planets[i],
                batched_angular_velocity[i],
                batched_comet_planet_ids[i],
                batch_time_steps[i],
                apply_padding=apply_padding
            )
            batch_states.append(state)
            if apply_padding:
                batch_masks.append(mask)
        
        # Stack into batch
        batch_states = np.stack(batch_states, axis=0)  # [batch_size, max_planets + max_fleets, feature_dim]
        
        if apply_padding:
            batch_masks = np.stack(batch_masks, axis=0)  # [batch_size, max_planets + max_fleets]
            return batch_states, batch_masks
        else:
            return batch_states, None


### Q_net

In [35]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np 

from utils.pos_encoding import get_pos_encoding

    
class Q_network(nn.Module):
    def __init__(self, 
                 state_dim=14, 
                 action_dim=8,
                 max_planets=40,
                 max_fleets=100, 
                 d_model=128, 
                 nhead=4, 
                 num_layers=3,  
                 dim_feedforward=2048, 
                 dropout=0.1):
        
        super(Q_network, self).__init__()
        self.max_planets = max_planets
        self.max_fleets = max_fleets

        self.P = nn.Linear(state_dim - 4, d_model)
        self.F = nn.Linear(state_dim - 4, d_model)
        self.A = nn.Linear(action_dim, d_model)

        self.value_head = nn.Linear(d_model, 1)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.sep_token = nn.Parameter(torch.zeros(1, 1, d_model))

        #transformer - encoder only
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True  # Recommended for easier tensor handling
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 
                                                 num_layers=num_layers, 
        )
        
        self.d_model = d_model
    

    def forward(self, state, action):  
        """
        Args:
            state: [batch_size, state_seq_len, d_model] or [state_seq_len, d_model]
            action: [batch_size, action_seq_len, d_model] or [action_seq_len, d_model]
        
        Returns:
            value: [batch_size] or [1] (scalar if unbatched)
        """
        #true for seq length upto max_planets
        planets_mask = torch.zeros(state.shape[0], state.shape[1])
        planets_mask[:, :self.max_planets] = 1

        fleets_mask = torch.zeros(state.shape[0], state.shape[1])
        fleets_mask[:, self.max_planets:self.max_planets + self.max_fleets] = 1

        # print(planets_mask[0]) 
        # print(fleets_mask[0])

        # print(f"planets mask shape {planets_mask.shape}, fleets mask shape {fleets_mask.shape}")
        #Usable dimension upto 10 
        planets = self.P(state[:, :, :10] * planets_mask.unsqueeze(-1))     
        fleets = self.F(state[:, :, :10] * fleets_mask.unsqueeze(-1))

        time_step = state[:, :, -1]  # Assuming time step is the last feature of the first token (planet)
        pos = state[:, :, 11:13]  # Assuming position is at indices 11 and 12
        pos_encoding = get_pos_encoding(pos, time_step, self.d_model)

        action = self.A(action)

        state = planets + fleets
        state = state + pos_encoding

        batch_size = state.shape[0]
        # state_n = state.shape[1]
        # action_n = action.shape[1]

        # Expand cls and sep tokens for batch
        cls_token = self.cls_token.expand(batch_size, -1, -1)  # [1, batch_size, d_model]
        sep_token = self.sep_token.expand(batch_size, -1, -1)  # [1, batch_size, d_model]

        # Concatenate: [batch_size, seq_len, d_model]
        src = torch.cat((cls_token, state,  sep_token, action), dim=1)

        out_ = self.transformer(src)
        out_cls = out_[:, 0, :]  # [batch_size, 1, d_model]
        value = self.value_head(out_cls)  # [batch_size, 1]
        
        return value.squeeze(-1)  # [batch_size]
    

class V_network(nn.Module):
    def __init__(self, 
                 state_dim=14,
                 max_planets=40,
                 max_fleets=100,
                 d_model=128, 
                 nhead=4, 
                 num_layers=2,  
                 dim_feedforward=128, 
                 dropout=0.1):
        
        super(V_network, self).__init__()
        self.max_planets = max_planets
        self.max_fleets = max_fleets
        
        self.P = nn.Linear(state_dim - 4, d_model)
        self.F = nn.Linear(state_dim - 4, d_model)

        self.value_head = nn.Linear(d_model, 1)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        #transformer - encoder only
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True  # Recommended for easier tensor handling
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 
                                                 num_layers=num_layers, 
        )
        self.d_model = d_model

    def forward(self, state):  
        
        planets_mask = torch.zeros(state.shape[0], state.shape[1])
        planets_mask[:, :self.max_planets] = 1

        fleets_mask = torch.zeros(state.shape[0], state.shape[1])
        fleets_mask[:, self.max_planets:self.max_planets + self.max_fleets] = 1

        planets = self.P(state[:, :, :10] * planets_mask.unsqueeze(-1))     
        fleets = self.F(state[:, :, :10] * fleets_mask.unsqueeze(-1))

        time_step = state[:, :, -1]  # Assuming time step is the last feature of the first token (planet)
        pos = state[:, :, 11:13]  # Assuming position is at indices 11 and 12
        pos_encoding = get_pos_encoding(pos, time_step, self.d_model)

        state = planets + fleets
        state = state + pos_encoding
    
        # Expand cls token for batch
        batch_size = state.shape[0]
        cls_token = self.cls_token.expand(batch_size, -1,  -1)  # [1, batch_size, d_model]

        src = torch.cat((cls_token, state), dim=1)
        out_ = self.transformer(src)
        out_cls = out_[:, 0, :]  # [batch_size, 1, d_model]
        value = self.value_head(out_cls)  # [batch_size, 1]
        
        return value.squeeze(-1)  # [batch_size]
    

class P_network(nn.Module):
    def __init__(self, 
                 d_model=128, 
                 state_dim=14,
                 action_dim=8,
                 max_planets=40,
                 max_fleets=100, 
                 nhead=4, 
                 num_layers=3,  
                 dim_feedforward=128, 
                 dropout=0.1):
        
        super(P_network, self).__init__()

        self.max_planets = max_planets
        self.max_fleets = max_fleets

        self.P = nn.Linear(state_dim - 4, d_model)
        self.F = nn.Linear(state_dim - 4, d_model)

        self.mu_head = nn.Linear(d_model, action_dim)
        self.sigma_head = nn.Linear(d_model, action_dim)

        #transformer - encoder only
        encoder_layer = nn.TransformerEncoderLayer(
                            d_model=d_model, 
                            nhead=nhead, 
                            dim_feedforward=dim_feedforward,
                            dropout=dropout,
                            activation='relu',
                            batch_first=True  # Recommended for easier tensor handling
                        )
        self.transformer = nn.TransformerEncoder(encoder_layer, 
                                                 num_layers=num_layers, 
                        )
        
        self.d_model = d_model
        self.action_dim = action_dim

    def forward(self, state):  
        planets_mask = torch.zeros(state.shape[0], state.shape[1])
        planets_mask[:, :self.max_planets] = 1

        fleets_mask = torch.zeros(state.shape[0], state.shape[1])
        fleets_mask[:, self.max_planets:self.max_planets + self.max_fleets] = 1

        planets = self.P(state[:, :, :10] * planets_mask.unsqueeze(-1))     
        fleets = self.F(state[:, :, :10] * fleets_mask.unsqueeze(-1))
        
        time_step = state[:, :, -1]  # Assuming time step is the last feature of the first token (planet)
        pos = state[:, :, 11:13]  # Assuming position is at indices 11 and 12
        pos_encoding = get_pos_encoding(pos, time_step, self.d_model)

        state = planets + fleets
        state = state + pos_encoding
    
        # Process through transformer
        src = state  # [batch_size, seq_len, d_model]
        out_ = self.transformer(src)  # [batch_size, seq_len, d_model]
        
        #only pass num planets tokens through heads, as action is only for planets
        mu = self.mu_head(out_[:, :self.max_planets, :])  
        sigma = F.softplus(self.sigma_head(out_[:, :self.max_planets, :])) + 1e-5  
        
        return mu, sigma
    
class ActionDecoder(nn.Module):
    def __init__(self, 
                 d_action=64):  
        super(ActionDecoder, self).__init__()
        self.d_action = d_action 

    def sample_action(self, mu, sigma):
        #gaussian sampling with reparameterization trick
        eps = torch.randn_like(sigma)
        action = mu + eps * sigma  # shape: [num_planets, action_dim]
        return action
    
    def forward(self, mu, sigma, mask=None):
        action = self.sample_action(mu, sigma)
        action = torch.mm(action, action.T)  # shape: [num_planets, num_planets]
        #softmax along rows to get probabilities of actions for each planet
        action = F.softmax(action, dim=1)

        #mask diagnols
        diag_mask = torch.eye(action.shape[0])
        action = action * (1 - diag_mask)
        
        if mask is not None:
            action = action * mask


        return action

## Test

In [ ]:
# Test batch encoding
MAX_PLANETS = 44 # max planets = 40 +  4 comets 
MAX_FLEETS = 100

batch_size = 4
start_idx = 10
idx_range = range(start_idx, start_idx + batch_size)

batched_fleets = [np.array(logs['steps'][i][0]['observation']['fleets']) for i in idx_range]
batched_planets = [np.array(logs['steps'][i][0]['observation']['planets']) for i in idx_range]
batched_comet_planet_ids = [np.array(logs['steps'][i][0]['observation']['comet_planet_ids']) for i in idx_range]
batched_angular_velocity = np.array([logs['steps'][i][0]['observation']['angular_velocity'] for i in idx_range])
batched_initial_planets = [np.array(logs['steps'][i][0]['observation']['initial_planets']) for i in idx_range]

encoder = Encoder(max_planets=MAX_PLANETS, max_fleets=MAX_FLEETS, max_comet_planet_ids=10)

# Process batch - returns [B, max_planets + max_fleets, feature_dim] and [B, max_planets + max_fleets]
batch_states, batch_masks = encoder.encode_batch(
    batched_planets,
    batched_fleets,
    batched_initial_planets,
    batched_angular_velocity,
    batched_comet_planet_ids,
    batch_time_steps=100
)

print(f"Batch states shape: {batch_states.shape}")
print(f"Batch masks shape: {batch_masks.shape}")
print(f"Expected states: [batch_size={batch_size}, max_planets + max_fleets={encoder.max_planets + encoder.max_fleets}, feature_dim]")
print(f"Expected masks: [batch_size={batch_size}, max_planets + max_fleets={encoder.max_planets + encoder.max_fleets}]")


Batch states shape: (4, 144, 14)
Batch masks shape: (4, 144)
Expected states: [batch_size=4, max_planets + max_fleets=144, feature_dim]
Expected masks: [batch_size=4, max_planets + max_fleets=144]


In [46]:
q_network = Q_network(max_fleets=MAX_FLEETS, max_planets=MAX_PLANETS)

batch_states_torch = torch.from_numpy(batch_states).float()
dummy_actions = torch.zeros(batch_size, MAX_PLANETS, 8)  # [B, num_actions, d_model]
q_values = q_network(batch_states_torch, dummy_actions)
print(f"Q-values shape: {q_values.shape}")  # Should be [B]

Q-values shape: torch.Size([4])


In [47]:
v_network = V_network(max_fleets=MAX_FLEETS, max_planets=MAX_PLANETS)
v_values = v_network(batch_states_torch)
print(f"V-values shape: {v_values.shape}")  # Should be [B]

V-values shape: torch.Size([4])


In [48]:
p_network = P_network(max_fleets=MAX_FLEETS, max_planets=MAX_PLANETS)
mu, sigma = p_network(batch_states_torch)
print(f"Policy mu shape: {mu.shape}, sigma shape: {sigma.shape}")  # Should be [B, seq_len, action_dim]

Policy mu shape: torch.Size([4, 44, 8]), sigma shape: torch.Size([4, 44, 8])


In [26]:
# Test encoding with empty fleets and comet_planet_ids
encoder = Encoder()

# Test case 1: Empty fleets
planets_test = np.array(logs['steps'][idx][0]['observation']['planets'])
fleets_test_empty = np.empty((0, 7))
comet_planet_ids_test = np.array(logs['steps'][idx][0]['observation']['comet_planet_ids'])
angular_velocity_test = logs['steps'][idx][0]['observation']['angular_velocity']
initial_planets_test = np.array(logs['steps'][idx][0]['observation']['initial_planets'])

state_empty_fleets, mask_empty_fleets = encoder.encode(
    planets_test, fleets_test_empty, initial_planets_test, 
    angular_velocity_test, comet_planet_ids_test, time_step=100
)

print("Test 1: Empty fleets")
print(f"  State shape: {state_empty_fleets.shape}")
print(f"  Mask shape: {mask_empty_fleets.shape}")
print(f"  Planets in mask (should be ~{len(planets_test)}): {mask_empty_fleets[:encoder.max_planets].sum()}")
print(f"  Fleets in mask (should be 0): {mask_empty_fleets[encoder.max_planets:].sum()}")

# Test case 2: Empty comet_planet_ids
fleets_test = np.array(logs['steps'][idx][0]['observation']['fleets'])
comet_planet_ids_test_empty = np.empty(0)

state_empty_comets, mask_empty_comets = encoder.encode(
    planets_test, fleets_test, initial_planets_test, 
    angular_velocity_test, comet_planet_ids_test_empty, time_step=100
)

print("\nTest 2: Empty comet_planet_ids")
print(f"  State shape: {state_empty_comets.shape}")
print(f"  Mask shape: {mask_empty_comets.shape}")
print(f"  Planets in mask: {mask_empty_comets[:encoder.max_planets].sum()}")
print(f"  Fleets in mask: {mask_empty_comets[encoder.max_planets:].sum()}")

# Test case 3: Both empty
state_both_empty, mask_both_empty = encoder.encode(
    planets_test, fleets_test_empty, initial_planets_test, 
    angular_velocity_test, comet_planet_ids_test_empty, time_step=100
)

print("\nTest 3: Both empty fleets and comet_planet_ids")
print(f"  State shape: {state_both_empty.shape}")
print(f"  Mask shape: {mask_both_empty.shape}")
print(f"  Planets in mask: {mask_both_empty[:encoder.max_planets].sum()}")
print(f"  Fleets in mask: {mask_both_empty[encoder.max_planets:].sum()}")


Test 1: Empty fleets
  State shape: (240, 14)
  Mask shape: (240,)
  Planets in mask (should be ~32): 32.0
  Fleets in mask (should be 0): 0.0

Test 2: Empty comet_planet_ids
  State shape: (240, 14)
  Mask shape: (240,)
  Planets in mask: 32.0
  Fleets in mask: 0.0

Test 3: Both empty fleets and comet_planet_ids
  State shape: (240, 14)
  Mask shape: (240,)
  Planets in mask: 32.0
  Fleets in mask: 0.0
